### Técnicas de treino — ablação

Mede o efeito de cada técnica de `mimo_sys.training` sobre o baseline no split 70/15/15 (SEED=42), com R² e energia por banda do forecast encadeado de 12 passos na escala original. Metodologia em `docs/6_tecnicas-de-treino.md`.

In [1]:
import json
import subprocess
import sys

import mlflow
import pandas as pd

from mimo_sys.training.data import OUTPUT_COLS

EXPERIMENT = "mimo_sys_training_techniques"
MODELS = {
    "seq2seq_latent_gnn": {"hidden_size": 64},
    "mpnn": {"hidden_size": 64, "message_dim": 8},
}

/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cada variante muda só o que está no dict; o resto é o `TrainConfig` padrão (MinMax no treino, MSE, AdamW, plateau).

In [2]:
VARIANTS = {
    "baseline": {},
    "revin": {"revin": "standard"},
    "augment": {"augment": True},
    "mixup": {"mixup_alpha": 0.2},
    "huber": {"loss": "huber"},
    "spectral": {"spectral_weight": 0.1},
    "ema": {"ema_decay": 0.999},
    "cosine_clip": {"scheduler": "cosine_warmup", "gradient_clip_val": 1.0},
    "contrastive": {"pretrain_epochs": 30, "freeze_encoder_epochs": 10},
    "combinado": {
        "revin": "standard",
        "augment": True,
        "spectral_weight": 0.1,
        "ema_decay": 0.999,
    },
}
# MPNN só tem encoder para pré-treinar com 2 camadas.
EXTRA_KWARGS = {("mpnn", "contrastive"): {"n_layers": 2}}

Um processo por treino: treinar vários modelos no mesmo processo já estourou a RAM (`docs/2_split_artigo_vs_dataset_completo.md`).

In [3]:
SCRIPT = """
import json, sys
from mimo_sys.training import TrainConfig, train
cfg = TrainConfig(**json.loads(sys.argv[1]))
print("RUN_ID", train(cfg).run_id)
"""


def run_variant(model: str, variant: str) -> str:
    """Treina uma variante num subprocesso e devolve o run_id do MLflow."""
    kwargs = {**MODELS[model], **EXTRA_KWARGS.get((model, variant), {})}
    cfg = {
        "model": model,
        "model_kwargs": kwargs,
        "experiment": EXPERIMENT,
        "run_name": f"{model}__{variant}",
        "save": False,
        **VARIANTS[variant],
    }
    out = subprocess.run(
        [sys.executable, "-c", SCRIPT, json.dumps(cfg)],
        capture_output=True,
        text=True,
        check=True,
    )
    return out.stdout.split("RUN_ID")[-1].strip()


run_ids = {
    (model, variant): run_variant(model, variant)
    for model in MODELS
    for variant in VARIANTS
}

### R² por saída (forecast encadeado, teste)

In [4]:
mlflow.set_experiment(EXPERIMENT)
runs = mlflow.search_runs(experiment_names=[EXPERIMENT]).set_index("run_id")

rows = []
for (model, variant), run_id in run_ids.items():
    run = runs.loc[run_id]
    row = {"modelo": model, "variante": variant}
    for out in OUTPUT_COLS:
        row[f"r2_{out}"] = run[f"metrics.chained_r2_{out}"]
    row["val_mse"] = run["metrics.best_val_mse"]
    rows.append(row)
r2 = pd.DataFrame(rows).set_index(["modelo", "variante"])
r2["r2_medio"] = r2.filter(like="r2_").mean(axis=1)
r2.round(3)

r2_vazao_distribuicao  r2_nivel_res  \
modelo             variante                                           
seq2seq_latent_gnn baseline                     0.881         0.325   
                   revin                        0.843         0.307   
                   augment                      0.876         0.344   
                   mixup                        0.873         0.343   
                   huber                        0.882         0.293   
                   spectral                     0.881         0.387   
                   ema                          0.883         0.305   
                   cosine_clip                  0.860         0.187   
                   contrastive                  0.882         0.295   
                   combinado                    0.895         0.387   
mpnn               baseline                     0.909         0.371   
                   revin                        0.922         0.379   
                   augment                      0.900         0.353   
                   mixup                        0.913         0.304   
                   huber                        0.916         0.210   
                   spectral                     0.891         0.234   
                   ema                          0.918         0.338   
                   cosine_clip                  0.902         0.433   
                   contrastive                  0.929         0.319   
                   combinado                    0.908         0.357   

                                r2_pressao  val_mse  r2_medio  
modelo             variante                                    
seq2seq_latent_gnn baseline          0.648    0.003     0.618  
                   revin             0.626    0.003     0.592  
                   augment           0.648    0.002     0.623  
                   mixup             0.633    0.003     0.616  
                   huber             0.646    0.002     0.607  
                   spectral          0.643    0.002     0.637  
                   ema               0.655    0.002     0.615  
                   cosine_clip       0.581    0.003     0.542  
                   contrastive       0.710    0.003     0.629  
                   combinado         0.692    0.003     0.658  
mpnn               baseline          0.667    0.003     0.649  
                   revin             0.659    0.003     0.653  
                   augment           0.652    0.002     0.635  
                   mixup             0.674    0.003     0.631  
                   huber             0.674    0.003     0.600  
                   spectral          0.639    0.003     0.588  
                   ema               0.667    0.002     0.641  
                   cosine_clip       0.665    0.003     0.667  
                   contrastive       0.699    0.003     0.649  
                   combinado         0.641    0.003     0.635

Ganho de R² médio sobre o baseline de cada arquitetura.

In [5]:
delta = r2["r2_medio"] - r2["r2_medio"].xs("baseline", level="variante")
delta.unstack("modelo").round(3).sort_values("seq2seq_latent_gnn")

modelo,mpnn,seq2seq_latent_gnn
variante,,
cosine_clip,0.018,-0.076
revin,0.004,-0.026
huber,-0.049,-0.011
ema,-0.008,-0.003
mixup,-0.018,-0.001
baseline,0.000,0.000
augment,-0.014,0.005
contrastive,0.000,0.011
spectral,-0.061,0.019


### Energia nas bandas altas

Erro relativo de energia (%) nas bandas abaixo do ciclo diário, onde o baseline perde energia; mais perto de 0 é melhor.

In [6]:
HIGH = ["harmonico_8-16h", "curto_4-8h", "ruido_2-4h"]
energy = []
for (model, variant), run_id in run_ids.items():
    run = runs.loc[run_id]
    for band in HIGH:
        for out in OUTPUT_COLS:
            energy.append({
                "modelo": model,
                "variante": variant,
                "banda": band,
                "saida": out,
                "energia_rel_%": run[
                    f"metrics.energia_rel_pct_{band}_{out}"
                ],
            })
energy = pd.DataFrame(energy)
energy.pivot_table(
    index=["modelo", "variante"],
    columns="banda",
    values="energia_rel_%",
    aggfunc="mean",
)[HIGH].round(1)

banda                           harmonico_8-16h  curto_4-8h  ruido_2-4h
modelo             variante                                            
mpnn               augment                -47.2       -59.5       -16.8
                   baseline               -51.1       -61.9         5.4
                   combinado              -46.8       -47.7        79.6
                   contrastive            -54.3       -58.3       -13.3
                   cosine_clip            -49.0       -53.1       -12.1
                   ema                    -50.3       -49.4        -2.5
                   huber                  -48.2       -52.9        15.8
                   mixup                  -47.8       -62.9         0.6
                   revin                  -45.5       -43.7        15.5
                   spectral               -45.0       -38.7       113.9
seq2seq_latent_gnn augment                -49.5       -51.5       -10.4
                   baseline               -50.4       -58.9        23.3
                   combinado              -40.9       -39.1       -28.0
                   contrastive            -52.4       -49.7        16.2
                   cosine_clip            -55.6       -36.6       307.4
                   ema                    -53.0       -63.1        16.1
                   huber                  -49.5       -58.5        29.0
                   mixup                  -52.9       -66.5        23.9
                   revin                  -42.5       -54.9       -55.9
                   spectral               -52.3       -48.6        -3.6